# Zalo AI Challenge 2026 — DataMate
## Bước 1: Sinh và lọc dataset SFT với Qwen2.5-Coder-7B-Instruct

Phiên bản 7B của `01_generate_sft_dataset_qwen14b.ipynb`, dùng chung
`scripts/generate_sft_vllm.py`. Chạy trên **Colab A100**, một GPU, BF16 như bản gốc.
Model: [Qwen/Qwen2.5-Coder-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct).

1. Model sinh lời giải/code từ câu hỏi, schema và ngữ cảnh nghiệp vụ của tập train.
2. Chạy code trong sandbox và đối chiếu với `data/train/answers.jsonl`
   qua bộ chấm nội bộ `score_single_answer`; chỉ giữ mẫu được chấm `correct` (+1).
3. Sinh theo batch, hỗ trợ observation/sửa lỗi tối đa 2 lượt, resume, streaming save và W&B.
4. Lưu dataset và log riêng ở `result/sft_dataset_qwen7b_folder_schema_v1/`;
   backup ở `/content/drive/MyDrive/datamate_sft_backup_qwen7b_folder_schema_v1/` khi Drive đã mount.

**Chuẩn bị:** đưa repo có bản script mới lên runtime cùng dữ liệu train.
Notebook gọi script trong repo, không nhúng source. `data/` luôn chỉ đọc.
Đây là bước tạo dataset, chưa phải fine-tune model. Tốc độ và tỷ lệ mẫu được giữ
cần đo khi chạy thực tế; notebook này không đưa ra số benchmark.


Đầu vào v1: BUSINESS_DIR + README + file/sheet + toàn bộ cột/dtype + câu hỏi + as_of_date. Không đưa dòng dữ liệu mẫu, unique values, full rulebook hay checklist lặp vào prompt. Code đọc quy tắc từ RULEBOOK_PATH khi cần. Ghi kết quả vào thư mục mới để giữ bộ dữ liệu cũ.

### 1. Cài đặt thư viện phụ thuộc (Dependencies)

In [ ]:
# Cài đặt các thư viện cần thiết cho DataMate, vLLM, Transformers và WandB
!pip install -q "pandas>=2.2.0" numpy openpyxl python-calamine xlrd duckdb pyyaml tqdm pydantic
!pip install -q torch transformers accelerate peft bitsandbytes wandb vllm


### 2. Thiết lập Môi trường & Kết nối Google Drive / Repo

In [ ]:
import os
import sys
from pathlib import Path

# Tự động nhận diện Google Colab và Mount Drive để lưu trữ dữ liệu an toàn
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Tìm kiếm thư mục project trên Drive hoặc /content
    candidates = [
        Path('/content/datamate'),
        Path('/content/ZALO-AI-CHALLENGE-2026'),
        Path('/content/drive/MyDrive/ZALO-AI-CHALLENGE-2026'),
        Path('/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026'),
        Path('/content'),
    ]
    PROJECT_ROOT = Path('/content')
    for cand in candidates:
        if cand.exists() and (cand / "data").exists():
            PROJECT_ROOT = cand
            break
else:
    PROJECT_ROOT = Path('.').resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Chuyển thư mục làm việc về PROJECT_ROOT
os.chdir(str(PROJECT_ROOT))

print(f"Thư mục làm việc: {PROJECT_ROOT}")
print(f"Đã tìm thấy thư mục data/train: {(PROJECT_ROOT / 'data' / 'train').exists()}")
print(f"CUDA có sẵn: {os.system('nvidia-smi') == 0}")


### 3. Đăng nhập Weights & Biases (Tùy chọn theo dõi trực quan)

In [ ]:
import wandb

wandb_api_key = os.environ.get("WANDB_API_KEY")
if not wandb_api_key and IN_COLAB:
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
    except Exception:
        pass

if wandb_api_key:
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login(key=wandb_api_key)
    print("✅ Logged in to Weights & Biases via API Key.")
else:
    print("ℹ️ Chưa cấu hình WANDB_API_KEY. Pipeline vẫn chạy và xuất log JSONL nội bộ bình thường.")


### 4. Kiểm tra & Tự động Làm sạch VRAM GPU (Khuyến nghị)
> 💡 **Mẹo:** Nếu trước đó bạn đã từng nạp model qua PyTorch/HuggingFace mà chưa khởi động lại session, GPU có thể vẫn bị chiếm dụng VRAM. Cell dưới đây sẽ tự động kiểm tra và giải phóng sạch GPU trước khi khởi chạy vLLM.

In [ ]:
import torch

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name} (Tổng VRAM: {total_vram:.1f} GB)")
    print(f"VRAM đang dùng (Allocated): {allocated:.2f} GB | Reserved: {reserved:.2f} GB")
    
    if allocated > 2.0:
        print("⚠️ VRAM đang bị chiếm dụng bởi model cũ! Đang tự động làm sạch kernel...")
        import os
        os.kill(os.getpid(), 9)
    else:
        print("✅ GPU hoàn toàn sạch sẽ, sẵn sàng nạp vLLM!")
else:
    print("❌ Không tìm thấy GPU CUDA! Vui lòng chọn Runtime -> Change runtime type -> A100 GPU.")


### 5. Sinh dataset theo batch, tối đa 2 lượt

- LLM sinh code theo batch; sandbox dùng tối đa 2 process CPU cùng lúc,
  mỗi process giới hạn 35 giây và một luồng tính toán để giảm tranh CPU/RAM.
- Script kiểm tra Calamine ngay khi khởi động. Nếu thiếu thư viện, script dừng
  trước khi nạp model; engine mặc định cho `.xlsx` là Calamine.
- Turn 2 chỉ nhận observation hoặc lỗi chạy/protocol/code; đáp án chuẩn chỉ
  dùng để lọc kết quả sau cùng. Mẫu được giữ lưu đủ lịch sử hội thoại.
- Prompt vượt ngân sách context được ghi log và bỏ qua, không cắt đuôi prompt.
- Sau mỗi batch sẽ in `LLM`, `sandbox_wall`, `total`, `kept`, `retry`,
  `timeouts`, `truncated` và `context_overflow`. Tốc độ cần đo trên runtime thật.

Notebook gọi source bên ngoài: trước khi chạy lại, cell chạy sẽ cập nhật các file source cần thiết từ thư mục dự án trên Drive nếu workspace đang ở local.
Dừng lệnh đang chạy rồi khởi chạy lệnh mới; thay file không cập nhật process đang chạy.
Giữ nguyên file output trong đúng thư mục để resume các ID đã được giữ.


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

# Refresh only source/config code; keep datasets and previous runs intact.
source_root = Path('/content/drive/MyDrive/Colab Notebooks/ZALO-AI-CHALLENGE-2026')
for relative in [
    'scripts/generate_sft_vllm.py',
    'src/agent/prompts.py',
    'src/agent/solver.py',
    'src/profiler/business_loader.py',
    'src/time_resolver/calendar_utils.py',
]:
    source, target = source_root / relative, PROJECT_ROOT / relative
    if source.is_file() and source.resolve() != target.resolve():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)

command = [
    sys.executable, '-u', 'scripts/generate_sft_vllm.py',
    '--model-id', 'Qwen/Qwen2.5-Coder-7B-Instruct',
    '--output-dir', 'result/sft_dataset_qwen7b_folder_schema_v1',
    '--drive-backup-dir', '/content/drive/MyDrive/datamate_sft_backup_qwen7b_folder_schema_v1',
    '--batch-size', '32', '--gpu-util', '0.88',
    '--max-model-len', '16384', '--max-tokens', '2048',
    '--sandbox-workers', '2', '--sandbox-timeout', '35',
]
with subprocess.Popen(command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace') as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if return_code:
    raise RuntimeError(f'Stage 1 failed (exit {return_code}); see output above.')


### 6. Kiểm tra & Thống kê Dataset SFT Thu Được

In [ ]:
import json
from pathlib import Path

sft_file = PROJECT_ROOT / "result" / "sft_dataset_qwen7b_folder_schema_v1" / "verified_sft_dataset.jsonl"
if not sft_file.exists():
    drive_file = Path("/content/drive/MyDrive/datamate_sft_backup_qwen7b_folder_schema_v1/verified_sft_dataset.jsonl")
    if drive_file.exists():
        sft_file = drive_file

if sft_file.exists():
    samples = []
    with open(sft_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
                
    print(f"🎉 Tổng số mẫu SFT đạt chuẩn (+1.0 điểm): {len(samples)} mẫu")
    print(f"📂 File lưu trữ: {sft_file} ({sft_file.stat().st_size / (1024**2):.2f} MB)\n")
    
    # In mẫu đầu tiên làm ví dụ kiểm tra Chat Template
    sample = samples[0]
    print(f"--- MẪU VÍ DỤ ({sample.get('id')} - {sample.get('business_id')}) ---")
    for msg in sample.get("messages", []):
        role = msg.get("role")
        content = msg.get("content", "")
        print(f"\n[{role.upper()}] ({len(content)} ký tự):")
        if role == "assistant":
            print(content)
        else:
            print(content[:300] + "...")
else:
    print(f"Chưa tìm thấy file {sft_file}. Hãy chạy cell 5 ở trên!")


### 7. Tự động ngắt kết nối session (tùy chọn)
Đặt `AUTO_SHUTDOWN = True` nếu muốn giải phóng session Colab sau khi sinh xong các câu đã chọn.


In [ ]:
AUTO_SHUTDOWN = False  # Đổi thành True nếu muốn tự động tắt máy sau khi xong

if AUTO_SHUTDOWN and IN_COLAB:
    print("🔌 Đang tự động ngắt kết nối session Colab để tiết kiệm Compute Units...")
    from google.colab import runtime
    runtime.unassign()
else:
    print("Session vẫn được duy trì.")
